In [ ]:
# AMD SEC Filing Workflow (10-Q / 10-K only)

Download AMD periodic filings, index them locally, and extract structured financial statements.

**Outputs** (written to `extracted/`):
- `filing_index.csv` — all local 10-Q / 10-K metadata
- `AMD_quarterly_financials.xlsx` — income statement, balance sheet, cash flow
- `AMD_annual_financials.xlsx` — annual versions of the same

Run cells top-to-bottom. Re-run the download cell only when you need new filings from SEC.

330it [02:41,  2.04it/s]                         
70it [00:49,  1.41it/s]                        
 92%|█████████▏| 1480/1610 [09:04<00:47,  2.72it/s]


ClientResponseError: 503, message='Service Unavailable', url='https://www.sec.gov/Archives/edgar/data/2488/0000898430-96-003346.txt'

In [9]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(r"e:\Quant\Quant\Investment Research\AMD").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from amd_sec_workflow import (
    FILINGS_DIR,
    OUTPUT_DIR,
    TICKER,
    build_filing_index,
    extract_financials,
    summarize,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ticker: {TICKER}")
print(f"Filings: {FILINGS_DIR}")
print(f"Output:  {OUTPUT_DIR}")

In [ ]:
# Optional: download / refresh 10-Q and 10-K from SEC.
# Skips files that already exist on disk. Safe to re-run after new earnings.

from amd_sec_workflow import download_filings

download_filings()
print("Download complete.")

Retrieved 5 periods of data


In [10]:
# Build filing index from local .txt files
filing_index = build_filing_index()
summarize(filing_index)
filing_index.tail(10)

In [ ]:
# Extract structured financials (edgartools) → Excel
quarterly_path, annual_path = extract_financials()
print(f"Quarterly: {quarterly_path}")
print(f"Annual:    {annual_path}")

In [ ]:
# Preview key quarterly revenue and net income
import re

import pandas as pd

income = pd.read_excel(quarterly_path, sheet_name="Income Statement")
period_cols = [c for c in income.columns if re.match(r"\d{4}-\d{2}-\d{2}", str(c))]

keywords = ("revenue", "net income")
preview = income[
    income["label"].str.contains("|".join(keywords), case=False, na=False)
    & ~income["abstract"].fillna(False)
].drop_duplicates(subset=["label"])

preview[["label", *period_cols[:6]]]